<a href="https://colab.research.google.com/github/JessieMorozov/Exercise_Form_Detection/blob/main/6_2_Copy_of_updated5_20__processing_engineered_feature_mvp_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installs

In [ ]:


!apt-get -y install ffmpeg
!pip -q install yt-dlp mediapipe opencv-python pandas numpy matplotlib scikit-learn joblib gdown

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


#Imports

In [ ]:


import os
import re
import json
import shutil
import subprocess
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, classification_report, confusion_matrix
)

#Drive+Project Paths

In [ ]:


drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/physio_project_updated2026_05_18")
DATASET_ROOT = PROJECT_ROOT / "dataset"

METADATA_DIR = DATASET_ROOT / "metadata"
CLIPS_DIR = DATASET_ROOT / "clips"
EXPORTS_DIR = DATASET_ROOT / "exports"

MASTER_INDEX_PATH = METADATA_DIR / "master_index.csv"

ARTIFACT_DIR = EXPORTS_DIR / "features"
MODEL_DIR = EXPORTS_DIR / "trained_models"
VECTORS_ZIP_DIR = EXPORTS_DIR / "vectors_zip"

for p in [METADATA_DIR, CLIPS_DIR, ARTIFACT_DIR, MODEL_DIR, VECTORS_ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Master index exists:", MASTER_INDEX_PATH.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/physio_project_updated2026_05_18
Master index exists: True


#Load + Audit Master Index

In [ ]:


def load_master_index():
    if not MASTER_INDEX_PATH.exists():
        raise FileNotFoundError(f"Missing master index: {MASTER_INDEX_PATH}")
    return pd.read_csv(MASTER_INDEX_PATH)


def dataset_summary(df=None):
    if df is None:
        df = load_master_index()

    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total clips: {len(df)}")

    for col in ["view_profile", "good_form", "exercise_type", "load_type", "label_confidence", "view_confidence", "usable"]:
        if col in df.columns:
            print(f"\n{col.upper()} COUNTS")
            print("-" * 40)
            print(df[col].fillna("MISSING").value_counts().sort_index())

    if {"view_profile", "good_form"}.issubset(df.columns):
        print("\nVIEW x LABEL TABLE")
        print("-" * 40)
        display(pd.crosstab(df["view_profile"], df["good_form"], margins=True))

    return df


master_df = dataset_summary()
display(master_df.head())

DATASET SUMMARY
Total clips: 110

VIEW_PROFILE COUNTS
----------------------------------------
view_profile
back            7
back_left      13
back_right     13
front          12
front_left     17
front_right    16
left_side      19
right_side     13
Name: count, dtype: int64

GOOD_FORM COUNTS
----------------------------------------
good_form
0    57
1    53
Name: count, dtype: int64

EXERCISE_TYPE COUNTS
----------------------------------------
exercise_type
back_squat          106
bodyweight_squat      4
Name: count, dtype: int64

LOAD_TYPE COUNTS
----------------------------------------
load_type
barbell       103
bodyweight      4
safety          3
Name: count, dtype: int64

LABEL_CONFIDENCE COUNTS
----------------------------------------
label_confidence
high      21
low        4
medium    85
Name: count, dtype: int64

VIEW_CONFIDENCE COUNTS
----------------------------------------
view_confidence
high      10
low        6
medium    94
Name: count, dtype: int64

USABLE COUNTS
--

good_form,0,1,All
view_profile,,,
back,5,2,7
back_left,6,7,13
back_right,7,6,13
front,4,8,12
front_left,7,10,17
front_right,9,7,16
left_side,12,7,19
right_side,7,6,13
All,57,53,110


,clip_id,url,source_platform,start_time,end_time,view_profile,view_angle_notes,exercise_type,load_type,good_form,label_confidence,view_confidence,usable,notes,start_sec,end_sec
0,clip_0001,https://www.youtube.com/watch?v=UMOYXKux1EI,youtube,0:27,0:33,left_side,mostly left side; slight front angle,back_squat,barbell,0,high,medium,1,Visible knee/hip profile.,27,33
1,clip_0002,https://www.youtube.com/shorts/PPmvh7gBTi0,youtube,0:31,0:33,left_side,short clip; side profile,back_squat,barbell,1,high,high,1,Very short time window.,31,33
2,clip_0003,https://www.youtube.com/shorts/6LnALQ338Ws,youtube,0:32,0:34,front_left,short clip; side profile,back_squat,barbell,1,medium,medium,1,Very short time window.,32,34
3,clip_0004,https://www.youtube.com/shorts/AscZRiHwHFs,youtube,0:12,0:17,left_side,straight side,back_squat,barbell,0,medium,medium,1,HAS TEXT CAREFUL,12,17
4,clip_0005,https://www.youtube.com/shorts/Lq9bf_QUSns,youtube,0:06,0:08,front_left,NaN,back_squat,barbell,1,medium,high,1,VERY fast,6,8


#Timestamp string parsing

In [ ]:


def parse_timestamp_to_seconds(ts):
    """
    converts timestamps to seconds:
      33, "33", "0:33", "00:33", "4:38", "1:02:03"
    """
    if pd.isna(ts):
        raise ValueError("Timestamp is missing.")

    if isinstance(ts, (int, np.integer)):
        return int(ts)

    if isinstance(ts, (float, np.floating)) and float(ts).is_integer():
        return int(ts)

    s = str(ts).strip()
    if s == "":
        raise ValueError("Timestamp is blank.")

    if re.fullmatch(r"\d+(\.\d+)?", s):
        return int(float(s))

    parts = s.split(":")
    if not all(re.fullmatch(r"\d+(\.\d+)?", p.strip()) for p in parts):
        raise ValueError(f"Invalid timestamp format: {ts}")

    nums = [float(p) for p in parts]

    if len(nums) == 2:
        minutes, seconds = nums
        total = minutes * 60 + seconds
    elif len(nums) == 3:
        hours, minutes, seconds = nums
        total = hours * 3600 + minutes * 60 + seconds
    else:
        raise ValueError(f"Invalid timestamp format: {ts}")

    return int(round(total))


def seconds_to_timestamp(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    rem = seconds % 3600
    m = rem // 60
    s = rem % 60

    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


for x in ["33", "0:33", "00:33", "4:38", "1:02:03", 12, 12.0]:
    print(x, "->", parse_timestamp_to_seconds(x), "->", seconds_to_timestamp(parse_timestamp_to_seconds(x)))

33 -> 33 -> 0:33
0:33 -> 33 -> 0:33
00:33 -> 33 -> 0:33
4:38 -> 278 -> 4:38
1:02:03 -> 3723 -> 1:02:03
12 -> 12 -> 0:12
12.0 -> 12 -> 0:12


#std. master index times

In [ ]:


def standardize_master_timestamps(save=True):
    df = load_master_index().copy()

    df["start_sec"] = df["start_time"].apply(parse_timestamp_to_seconds)
    df["end_sec"] = df["end_time"].apply(parse_timestamp_to_seconds)

    bad = df[df["end_sec"] <= df["start_sec"]]
    if len(bad) > 0:
        print("Bad time windows found:")
        display(bad[["clip_id", "url", "start_time", "end_time", "start_sec", "end_sec"]])
        raise ValueError("Some clips have end_sec <= start_sec.")

    if save:
        df.to_csv(MASTER_INDEX_PATH, index=False)
        print("Saved standardized master_index.csv with start_sec/end_sec columns.")

    return df


master_df = standardize_master_timestamps(save=True)
display(master_df[["clip_id", "start_time", "end_time", "start_sec", "end_sec", "view_profile", "good_form"]].head())

Saved standardized master_index.csv with start_sec/end_sec columns.


,clip_id,start_time,end_time,start_sec,end_sec,view_profile,good_form
0,clip_0001,0:27,0:33,27,33,left_side,0
1,clip_0002,0:31,0:33,31,33,left_side,1
2,clip_0003,0:32,0:34,32,34,front_left,1
3,clip_0004,0:12,0:17,12,17,left_side,0
4,clip_0005,0:06,0:08,6,8,front_left,1


#Download/frame extraction strategy

Default behavior:

- download a temporary time segment
- exract frames
- run MediaPipe
- delerte raw video unless explicit choice made to keep it
- keep `pose_vectors.csv` as the durable artifact



##vid download helpser

In [ ]:


def run_command(cmd, verbose=True):
    if verbose:
        print("Running:", " ".join(str(x) for x in cmd))

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    if verbose:
        print(result.stdout[-2000:])

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

    return result.stdout


def download_clip_segment(url, start_sec, end_sec, out_path, max_height=720, overwrite=False):
    out_path = Path(out_path)

    if out_path.exists() and not overwrite:
        print(f"Raw video already exists, skipping download: {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    section = f"*{int(start_sec)}-{int(end_sec)}"

    cmd = [
        "yt-dlp",
        "-f", f"bv*[height<={max_height}]+ba/b[height<={max_height}]/best[height<={max_height}]/best",
        "--merge-output-format", "mp4",
        "--download-sections", section,
        "--force-keyframes-at-cuts",
        "-o", str(out_path),
        url,
    ]

    run_command(cmd, verbose=True)

    if not out_path.exists():
        candidates = sorted(out_path.parent.glob(out_path.stem + "*"))
        if candidates:
            candidates[0].rename(out_path)

    if not out_path.exists():
        raise FileNotFoundError(f"Download did not produce expected file: {out_path}")

    return out_path

##frame extraction helpers

In [ ]:


def clear_directory(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def extract_frames_from_video(video_path, frames_dir, sample_fps=10, overwrite=False, image_ext="jpg"):
    video_path = Path(video_path)
    frames_dir = Path(frames_dir)

    if frames_dir.exists() and any(frames_dir.glob(f"*.{image_ext}")) and not overwrite:
        existing = sorted(frames_dir.glob(f"*.{image_ext}"))
        print(f"Frames already exist, skipping extraction: {len(existing)} frames")
        return existing

    clear_directory(frames_dir)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    source_fps = cap.get(cv2.CAP_PROP_FPS)
    if source_fps is None or source_fps <= 0 or np.isnan(source_fps):
        source_fps = 30.0

    stride = max(1, int(round(source_fps / sample_fps)))

    frame_idx = 0
    saved_idx = 0
    saved_paths = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % stride == 0:
            out_name = f"frame_{saved_idx:05d}.{image_ext}"
            out_path = frames_dir / out_name
            cv2.imwrite(str(out_path), frame)
            saved_paths.append(out_path)
            saved_idx += 1

        frame_idx += 1

    cap.release()

    print(f"Extracted {len(saved_paths)} frames from {video_path.name}")
    return saved_paths

##Mediapipe model setup

In [ ]:


MODEL_PATH = Path("/content/pose_landmarker_lite.task")

def ensure_mediapipe_pose_model():
    if MODEL_PATH.exists():
        print("MediaPipe model already exists:", MODEL_PATH)
        return MODEL_PATH

    file_id = "19bjztgjGm-au_32ZvhuH7Bfy9HZPGXso"
    cmd = ["gdown", file_id, "-O", str(MODEL_PATH)]

    try:
        run_command(cmd, verbose=True)
    except Exception:
        print("gdown failed. Trying direct URL fallback...")
        url = f"https://drive.google.com/uc?id={file_id}"
        run_command(["wget", "-O", str(MODEL_PATH), url], verbose=True)

    if not MODEL_PATH.exists():
        raise FileNotFoundError("Could not download pose_landmarker_lite.task")

    return MODEL_PATH


MEDIAPIPE_IDXS = {
    "nose": 0,
    "left_eye_inner": 1,
    "left_eye": 2,
    "left_eye_outer": 3,
    "right_eye_inner": 4,
    "right_eye": 5,
    "right_eye_outer": 6,
    "left_ear": 7,
    "right_ear": 8,
    "mouth_left": 9,
    "mouth_right": 10,
    "left_shoulder": 11,
    "right_shoulder": 12,
    "left_elbow": 13,
    "right_elbow": 14,
    "left_wrist": 15,
    "right_wrist": 16,
    "left_pinky": 17,
    "right_pinky": 18,
    "left_index": 19,
    "right_index": 20,
    "left_thumb": 21,
    "right_thumb": 22,
    "left_hip": 23,
    "right_hip": 24,
    "left_knee": 25,
    "right_knee": 26,
    "left_ankle": 27,
    "right_ankle": 28,
    "left_heel": 29,
    "right_heel": 30,
    "left_foot_index": 31,
    "right_foot_index": 32,
}

##Pose detection + drawing helpers

In [ ]:


POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (15, 17), (15, 19), (15, 21),
    (16, 18), (16, 20), (16, 22),
    (11, 23), (12, 24),
    (23, 24),
    (23, 25), (25, 27), (27, 29), (29, 31),
    (24, 26), (26, 28), (28, 30), (30, 32),
    (27, 31), (28, 32),
]


def rotate_points(points, angle_rad):
    R = np.array([
        [np.cos(angle_rad), -np.sin(angle_rad)],
        [np.sin(angle_rad),  np.cos(angle_rad)]
    ], dtype=np.float32)
    return points @ R.T


def normalize_pose_2d(points, left_hip_idx=23, right_hip_idx=24, left_shoulder_idx=11, right_shoulder_idx=12):
    # normalizing 33x2 pixel coordinates:
    # 1. centering at hip midpoint
    # 2. rotating so hips are horizontal
    # 3. scaling by torso length if possible, else hip width
    points = np.asarray(points, dtype=np.float32).copy()

    left_hip = points[left_hip_idx]
    right_hip = points[right_hip_idx]
    hip_center = (left_hip + right_hip) / 2.0
    points -= hip_center

    hip_vector = right_hip - left_hip
    angle = np.arctan2(hip_vector[1], hip_vector[0])
    points = rotate_points(points, -angle)

    left_shoulder = points[left_shoulder_idx]
    right_shoulder = points[right_shoulder_idx]
    shoulder_center = (left_shoulder + right_shoulder) / 2.0
    torso_length = np.linalg.norm(shoulder_center)

    if torso_length > 1e-6:
        scale = torso_length
    else:
        hip_width = np.linalg.norm(points[right_hip_idx] - points[left_hip_idx])
        scale = hip_width if hip_width > 1e-6 else 1.0

    points /= scale
    return points


def landmark_vector(points):
    pts = np.asarray(points, dtype=np.float32)
    if pts.shape != (33, 2):
        raise ValueError(f"Expected pose shape (33,2), got {pts.shape}")
    return pts.reshape(-1)


def get_pose_landmarks_from_image(image_bgr, landmarker):
    h, w = image_bgr.shape[:2]

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    detection_result = landmarker.detect(mp_image)

    if not detection_result.pose_landmarks:
        return None, None, None

    pose_landmarks = detection_result.pose_landmarks[0]

    points_px = []
    visibility = []
    for lm in pose_landmarks:
        points_px.append([lm.x * w, lm.y * h])
        visibility.append(getattr(lm, "visibility", np.nan))

    return np.array(points_px, dtype=np.float32), np.array(visibility, dtype=np.float32), pose_landmarks


def draw_pose_landmarks_on_image(image_bgr, pose_landmarks_raw):
    annotated = image_bgr.copy()
    h, w = annotated.shape[:2]

    for lm in pose_landmarks_raw:
        x = int(lm.x * w)
        y = int(lm.y * h)
        cv2.circle(annotated, (x, y), 3, (0, 255, 0), -1)

    for start_idx, end_idx in POSE_CONNECTIONS:
        lm1 = pose_landmarks_raw[start_idx]
        lm2 = pose_landmarks_raw[end_idx]

        x1, y1 = int(lm1.x * w), int(lm1.y * h)
        x2, y2 = int(lm2.x * w), int(lm2.y * h)

        cv2.line(annotated, (x1, y1), (x2, y2), (255, 0, 0), 2)

    return annotated

##Extracting pose vectors from frames

In [ ]:


def extract_pose_vectors_from_frames(
    frames_dir,
    out_csv_path,
    model_path=MODEL_PATH,
    save_annotated=False,
    annotated_dir=None,
    overwrite=False,
):
    frames_dir = Path(frames_dir)
    out_csv_path = Path(out_csv_path)

    if out_csv_path.exists() and not overwrite:
        print(f"Vector CSV already exists, skipping: {out_csv_path}")
        return pd.read_csv(out_csv_path)

    if save_annotated:
        if annotated_dir is None:
            annotated_dir = frames_dir.parent / "annotated_frames"
        annotated_dir = Path(annotated_dir)
        clear_directory(annotated_dir)

    valid_exts = {".jpg", ".jpeg", ".png"}
    frame_paths = sorted([p for p in frames_dir.iterdir() if p.suffix.lower() in valid_exts])

    if len(frame_paths) == 0:
        raise ValueError(f"No frames found in {frames_dir}")

    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )

    rows = []

    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        for frame_idx, frame_path in enumerate(frame_paths):
            image_bgr = cv2.imread(str(frame_path))

            if image_bgr is None:
                continue

            points_px, visibility, raw_landmarks = get_pose_landmarks_from_image(image_bgr, landmarker)

            if points_px is None:
                continue

            points_norm = normalize_pose_2d(points_px)
            vec = landmark_vector(points_norm)

            row = {
                "frame_name": frame_path.name,
                "frame_idx": frame_idx,
            }

            for i, val in enumerate(vec):
                row[f"v{i}"] = float(val)

            for j, vis in enumerate(visibility):
                row[f"vis{j}"] = float(vis) if not np.isnan(vis) else np.nan

            rows.append(row)

            if save_annotated:
                annotated = draw_pose_landmarks_on_image(image_bgr, raw_landmarks)
                cv2.imwrite(str(annotated_dir / frame_path.name), annotated)

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError(f"MediaPipe found no usable poses in {frames_dir}")

    out_csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv_path, index=False)

    print(f"Saved {len(df)} pose-vector rows to {out_csv_path}")
    return df

##Process One clip from registry

In [ ]:


def process_one_clip_from_registry(
    clip_id,
    sample_fps=10,
    keep_raw_video=False,
    keep_frames=True,
    save_annotated=False,
    overwrite_download=False,
    overwrite_frames=False,
    overwrite_vectors=False,
):
    df = load_master_index()
    row_df = df[df["clip_id"] == clip_id]

    if len(row_df) != 1:
        raise ValueError(f"Expected exactly one row for {clip_id}, found {len(row_df)}")

    row = row_df.iloc[0].to_dict()

    clip_dir = CLIPS_DIR / clip_id
    frames_dir = clip_dir / "frames"
    raw_video_path = clip_dir / "raw_video.mp4"
    vector_csv_path = clip_dir / "pose_vectors.csv"
    annotated_dir = clip_dir / "annotated_frames"

    clip_dir.mkdir(parents=True, exist_ok=True)

    start_sec = int(row["start_sec"]) if "start_sec" in row and not pd.isna(row["start_sec"]) else parse_timestamp_to_seconds(row["start_time"])
    end_sec = int(row["end_sec"]) if "end_sec" in row and not pd.isna(row["end_sec"]) else parse_timestamp_to_seconds(row["end_time"])

    ensure_mediapipe_pose_model()

    download_clip_segment(
        url=row["url"],
        start_sec=start_sec,
        end_sec=end_sec,
        out_path=raw_video_path,
        overwrite=overwrite_download,
    )

    extract_frames_from_video(
        video_path=raw_video_path,
        frames_dir=frames_dir,
        sample_fps=sample_fps,
        overwrite=overwrite_frames,
    )

    vec_df = extract_pose_vectors_from_frames(
        frames_dir=frames_dir,
        out_csv_path=vector_csv_path,
        save_annotated=save_annotated,
        annotated_dir=annotated_dir,
        overwrite=overwrite_vectors,
    )

    if not keep_raw_video and raw_video_path.exists():
        raw_video_path.unlink()
        print("Deleted raw video to save space:", raw_video_path)

    if not keep_frames and frames_dir.exists():
        shutil.rmtree(frames_dir)
        print("Deleted frames to save space:", frames_dir)

    return vec_df


# Example:
process_one_clip_from_registry("clip_0001", sample_fps=10, save_annotated=True)

Running: gdown 19bjztgjGm-au_32ZvhuH7Bfy9HZPGXso -O /content/pose_landmarker_lite.task
Downloading...
From: https://drive.google.com/uc?id=19bjztgjGm-au_32ZvhuH7Bfy9HZPGXso
To: /content/pose_landmarker_lite.task

  0%|          | 0.00/5.78M [00:00<?, ?B/s]
 45%|████▌     | 2.62M/5.78M [00:00<00:00, 12.3MB/s]
100%|██████████| 5.78M/5.78M [00:00<00:00, 24.8MB/s]

Running: yt-dlp -f bv*[height<=720]+ba/b[height<=720]/best[height<=720]/best --merge-output-format mp4 --download-sections *27-33 --force-keyframes-at-cuts -o /content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0001/raw_video.mp4 https://www.youtube.com/watch?v=UMOYXKux1EI
=  125 fps= 37 q=29.0 size=     256kB time=00:00:02.53 bitrate= 827.1kbits/s speed=0.759x    
frame=  140 fps= 36 q=29.0 size=     512kB time=00:00:03.04 bitrate=1379.0kbits/s speed=0.79x    
frame=  154 fps= 35 q=29.0 size=     512kB time=00:00:03.50 bitrate=1196.4kbits/s speed=0.802x    
frame=  169 fps= 35 q=29.0 size=     512kB time=

,frame_name,frame_idx,v0,v1,v2,v3,v4,v5,v6,v7,...,vis23,vis24,vis25,vis26,vis27,vis28,vis29,vis30,vis31,vis32
0,frame_00003.jpg,3,0.340867,1.190870,0.312867,1.271089,0.288836,1.273950,0.264231,1.277197,...,0.998928,0.997654,0.618064,0.165441,0.633495,0.206587,0.514854,0.350268,0.676643,0.326769
1,frame_00011.jpg,11,-0.085223,-1.316016,-0.061706,-1.335803,-0.065770,-1.331009,-0.069760,-1.326480,...,0.998756,0.998068,0.894493,0.105539,0.884726,0.168247,0.855770,0.291192,0.885247,0.268417
2,frame_00012.jpg,12,0.221037,-1.366529,0.276157,-1.361793,0.263170,-1.356144,0.250375,-1.350598,...,0.999286,0.998838,0.954120,0.172921,0.925382,0.256829,0.906564,0.273836,0.937140,0.381684
3,frame_00013.jpg,13,0.765471,-1.070385,0.875487,-1.060283,0.879575,-1.049016,0.883927,-1.037848,...,0.995849,0.996016,0.942847,0.291352,0.889231,0.361945,0.872140,0.377668,0.922072,0.497104
4,frame_00014.jpg,14,0.548402,-1.263987,0.579525,-1.267969,0.574504,-1.258279,0.570058,-1.248540,...,0.999386,0.999252,0.967394,0.196596,0.935924,0.279638,0.927203,0.326175,0.947439,0.410106
5,frame_00015.jpg,15,0.118169,-1.304720,0.151283,-1.347493,0.154032,-1.343690,0.157138,-1.339797,...,0.999127,0.998703,0.953144,0.129356,0.902862,0.197993,0.907622,0.331701,0.923636,0.318637
6,frame_00016.jpg,16,1.238813,0.318705,1.269760,0.364781,1.264230,0.371016,1.258739,0.377693,...,0.996818,0.997010,0.920728,0.202049,0.810581,0.229129,0.796176,0.304981,0.853758,0.329866
7,frame_00019.jpg,19,0.213864,1.186444,0.189690,1.270957,0.166580,1.275726,0.143087,1.280673,...,0.993312,0.994095,0.733566,0.733794,0.528464,0.698027,0.396514,0.558129,0.581178,0.715107
8,frame_00020.jpg,20,0.451206,1.136238,0.474686,1.205801,0.465388,1.209451,0.455270,1.213296,...,0.992739,0.993153,0.773344,0.715436,0.570919,0.659708,0.521949,0.547611,0.604309,0.668045
9,frame_00021.jpg,21,0.243926,1.169669,0.214081,1.285613,0.183889,1.295786,0.153214,1.305905,...,0.995743,0.996549,0.867841,0.815999,0.480959,0.537431,0.379270,0.480810,0.449690,0.486964


##Process many/all clips

In [ ]:


def process_all_registry_clips(
    only_usable=True,
    limit=None,
    sample_fps=10,
    keep_raw_video=False,
    keep_frames=True,
    save_annotated=False,
    overwrite_download=False,
    overwrite_frames=False,
    overwrite_vectors=False,
):
    df = load_master_index().copy()

    if only_usable and "usable" in df.columns:
        df = df[df["usable"].astype(str) != "0"]

    if limit is not None:
        df = df.head(limit)

    status_rows = []

    for _, row in df.iterrows():
        clip_id = row["clip_id"]
        print("\n" + "=" * 80)
        print(f"Processing {clip_id}")
        print("=" * 80)

        try:
            vec_df = process_one_clip_from_registry(
                clip_id=clip_id,
                sample_fps=sample_fps,
                keep_raw_video=keep_raw_video,
                keep_frames=keep_frames,
                save_annotated=save_annotated,
                overwrite_download=overwrite_download,
                overwrite_frames=overwrite_frames,
                overwrite_vectors=overwrite_vectors,
            )

            status_rows.append({
                "clip_id": clip_id,
                "status": "ok",
                "num_vector_frames": len(vec_df),
                "error": "",
            })

        except Exception as e:
            print(f"FAILED {clip_id}: {e}")
            status_rows.append({
                "clip_id": clip_id,
                "status": "failed",
                "num_vector_frames": 0,
                "error": str(e),
            })

    status_df = pd.DataFrame(status_rows)
    status_path = METADATA_DIR / "processing_status.csv"
    status_df.to_csv(status_path, index=False)

    print("\nSaved processing status:", status_path)
    display(status_df)

    return status_df


#small scale test
#status_df = process_all_registry_clips(limit=3, sample_fps=10, save_annotated=True)

#FULL RUN
status_df = process_all_registry_clips(sample_fps=10, save_annotated=False, keep_raw_video=False, keep_frames=True)


Processing clip_0001
MediaPipe model already exists: /content/pose_landmarker_lite.task
Running: yt-dlp -f bv*[height<=720]+ba/b[height<=720]/best[height<=720]/best --merge-output-format mp4 --download-sections *27-33 --force-keyframes-at-cuts -o /content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0001/raw_video.mp4 https://www.youtube.com/watch?v=UMOYXKux1EI
  133 fps= 34 q=29.0 size=     256kB time=00:00:02.80 bitrate= 746.6kbits/s speed=0.728x    
frame=  147 fps= 34 q=29.0 size=     512kB time=00:00:03.27 bitrate=1281.2kbits/s speed=0.748x    
frame=  162 fps= 33 q=29.0 size=     512kB time=00:00:03.77 bitrate=1112.5kbits/s speed=0.769x    
frame=  178 fps= 33 q=29.0 size=     512kB time=00:00:04.30 bitrate= 974.5kbits/s speed=0.792x    
frame=  180 fps= 27 q=-1.0 Lsize=     948kB time=00:00:05.99 bitrate=1297.0kbits/s speed=0.885x    
video:846kB audio:95kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.826975%
[libx264 @ 0x5672ad007f0

,clip_id,status,num_vector_frames,error
0,clip_0001,ok,42,
1,clip_0002,ok,20,
2,clip_0003,ok,20,
3,clip_0004,ok,30,
4,clip_0005,ok,25,
...,...,...,...,...
105,clip_0106,ok,29,
106,clip_0107,ok,75,
107,clip_0108,ok,19,
108,clip_0109,ok,40,


#Feature engineering

from here down, notebook assumes `pose_vectors.csv` exists for each clip. each CSV should have:

- one row per frame
- columns `v0` ... `v65` for 33 landmarks × 2 coordinates
- optional `vis0` ... `vis32` visibility columns

##vector csv loading + geometry

In [ ]:


def load_vector_csv_as_points(csv_path):
    df = pd.read_csv(csv_path)

    coord_cols = [f"v{i}" for i in range(66)]
    missing = [c for c in coord_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing vector columns in {csv_path}: {missing[:5]}...")

    points_seq = []

    for _, row in df.iterrows():
        vec = row[coord_cols].to_numpy(dtype=np.float32)
        if np.isnan(vec).any():
            continue
        points_seq.append(vec.reshape(33, 2))

    if len(points_seq) == 0:
        raise ValueError(f"No usable vector rows in {csv_path}")

    return points_seq, df


def angle_between_three_points(a, b, c):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    c = np.asarray(c, dtype=np.float32)

    ba = a - b
    bc = c - b

    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)

    if norm_ba < 1e-6 or norm_bc < 1e-6:
        return np.nan

    cos_angle = np.dot(ba, bc) / (norm_ba * norm_bc)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    return float(np.degrees(np.arccos(cos_angle)))


def torso_lean_deg(shoulder_center, hip_center):
    vec = np.asarray(shoulder_center) - np.asarray(hip_center)
    vertical = np.array([0.0, -1.0], dtype=np.float32)

    norm_vec = np.linalg.norm(vec)
    if norm_vec < 1e-6:
        return np.nan

    cos_angle = np.dot(vec, vertical) / norm_vec
    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    return float(np.degrees(np.arccos(cos_angle)))


def euclidean_distance(p1, p2):
    return float(np.linalg.norm(np.asarray(p2) - np.asarray(p1)))


def safe_divide(a, b, default=np.nan):
    if b is None or abs(b) < 1e-8:
        return default
    return float(a / b)

##frame-level engineered features

In [ ]:


def extract_frame_features(p, idxs=MEDIAPIPE_IDXS):
    p = np.asarray(p, dtype=np.float32)

    l_shoulder = p[idxs["left_shoulder"]]
    r_shoulder = p[idxs["right_shoulder"]]
    l_hip = p[idxs["left_hip"]]
    r_hip = p[idxs["right_hip"]]
    l_knee = p[idxs["left_knee"]]
    r_knee = p[idxs["right_knee"]]
    l_ankle = p[idxs["left_ankle"]]
    r_ankle = p[idxs["right_ankle"]]
    l_foot = p[idxs["left_foot_index"]]
    r_foot = p[idxs["right_foot_index"]]

    shoulder_center = (l_shoulder + r_shoulder) / 2.0
    hip_center = (l_hip + r_hip) / 2.0
    knee_center = (l_knee + r_knee) / 2.0
    ankle_center = (l_ankle + r_ankle) / 2.0

    left_knee_angle = angle_between_three_points(l_hip, l_knee, l_ankle)
    right_knee_angle = angle_between_three_points(r_hip, r_knee, r_ankle)
    left_hip_angle = angle_between_three_points(l_shoulder, l_hip, l_knee)
    right_hip_angle = angle_between_three_points(r_shoulder, r_hip, r_knee)

    left_femur = euclidean_distance(l_hip, l_knee)
    right_femur = euclidean_distance(r_hip, r_knee)
    left_tibia = euclidean_distance(l_knee, l_ankle)
    right_tibia = euclidean_distance(r_knee, r_ankle)
    torso_len = euclidean_distance(shoulder_center, hip_center)

    mean_femur = np.nanmean([left_femur, right_femur])
    mean_tibia = np.nanmean([left_tibia, right_tibia])

    hip_width = abs(r_hip[0] - l_hip[0])
    knee_width = abs(r_knee[0] - l_knee[0])
    ankle_width = abs(r_ankle[0] - l_ankle[0])

    knee_ankle_ratio = safe_divide(knee_width, ankle_width)
    valgus_score_global = 1.0 - knee_ankle_ratio if not np.isnan(knee_ankle_ratio) else np.nan

    left_knee_inward = max(0.0, l_knee[0] - l_ankle[0])
    right_knee_inward = max(0.0, r_ankle[0] - r_knee[0])

    ##ADD:
    #from total travel distance, define a section (proportion) as "near top" and "near bottom"
    #or, use minimum knee angle
    #within the near-bottom section, if hips travel from neutral/behind spine to in front of spine,
    #AND
    return {
        "left_knee_angle_deg": left_knee_angle,
        "right_knee_angle_deg": right_knee_angle,
        "mean_knee_angle_deg": np.nanmean([left_knee_angle, right_knee_angle]),
        "knee_angle_asym_deg": abs(left_knee_angle - right_knee_angle),

        "left_hip_angle_deg": left_hip_angle,
        "right_hip_angle_deg": right_hip_angle,
        "mean_hip_angle_deg": np.nanmean([left_hip_angle, right_hip_angle]),
        "hip_angle_asym_deg": abs(left_hip_angle - right_hip_angle),

        "torso_lean_deg": torso_lean_deg(shoulder_center, hip_center),

        "mean_femur_length": mean_femur,
        "mean_tibia_length": mean_tibia,
        "torso_length": torso_len,
        "femur_tibia_ratio": safe_divide(mean_femur, mean_tibia),
        "torso_femur_ratio": safe_divide(torso_len, mean_femur),
        "torso_tibia_ratio": safe_divide(torso_len, mean_tibia),
        "femur_length_asym": abs(left_femur - right_femur),
        "tibia_length_asym": abs(left_tibia - right_tibia),

        "hip_to_ankle_x": abs(hip_center[0] - ankle_center[0]),
        "knee_to_ankle_x": abs(knee_center[0] - ankle_center[0]),
        "shoulder_to_ankle_x": abs(shoulder_center[0] - ankle_center[0]),
        "hip_to_ankle_x_norm_femur": safe_divide(abs(hip_center[0] - ankle_center[0]), mean_femur),
        "knee_to_ankle_x_norm_tibia": safe_divide(abs(knee_center[0] - ankle_center[0]), mean_tibia),

        "hip_width": hip_width,
        "knee_width": knee_width,
        "ankle_width": ankle_width,
        "knee_ankle_ratio": knee_ankle_ratio,
        "valgus_score_global": valgus_score_global,
        "left_knee_inward_norm": safe_divide(left_knee_inward, ankle_width),
        "right_knee_inward_norm": safe_divide(right_knee_inward, ankle_width),
        "valgus_asymmetry": abs(
            safe_divide(left_knee_inward, ankle_width, default=0.0)
            - safe_divide(right_knee_inward, ankle_width, default=0.0)
        ),

        "stance_width_norm_torso": safe_divide(ankle_width, torso_len),
        "foot_width_norm_torso": safe_divide(abs(r_foot[0] - l_foot[0]), torso_len),
    }

##by-clip feature summary

In [ ]:


def summarize_series(series, prefix):
    arr = np.asarray(series, dtype=np.float32)
    arr = arr[~np.isnan(arr)]

    if arr.size == 0:
        return {f"{prefix}_{stat}": np.nan for stat in ["mean", "std", "min", "max", "range", "q25", "median", "q75", "slope"]}

    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr)),
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_max": float(np.max(arr)),
        f"{prefix}_range": float(np.max(arr) - np.min(arr)),
        f"{prefix}_q25": float(np.quantile(arr, 0.25)),
        f"{prefix}_median": float(np.quantile(arr, 0.50)),
        f"{prefix}_q75": float(np.quantile(arr, 0.75)),
        f"{prefix}_slope": float(arr[-1] - arr[0]) if arr.size >= 2 else 0.0,
    }


def compute_dynamic_summary(frame_df, col):
    arr = frame_df[col].to_numpy(dtype=np.float32)
    arr = arr[~np.isnan(arr)]

    if len(arr) < 2:
        return {
            f"{col}_velocity_mean_abs": np.nan,
            f"{col}_velocity_max_abs": np.nan,
            f"{col}_accel_mean_abs": np.nan,
            f"{col}_accel_max_abs": np.nan,
        }

    velocity = np.diff(arr)
    acceleration = np.diff(velocity)

    return {
        f"{col}_velocity_mean_abs": float(np.mean(np.abs(velocity))),
        f"{col}_velocity_max_abs": float(np.max(np.abs(velocity))),
        f"{col}_accel_mean_abs": float(np.mean(np.abs(acceleration))) if len(acceleration) else np.nan,
        f"{col}_accel_max_abs": float(np.max(np.abs(acceleration))) if len(acceleration) else np.nan,
    }


def compute_rule_based_measurements(frame_df):
    out = {}

    out["rule_peak_torso_lean_deg"] = float(frame_df["torso_lean_deg"].max())
    out["rule_min_mean_knee_angle_deg"] = float(frame_df["mean_knee_angle_deg"].min())
    out["rule_max_knee_angle_asym_deg"] = float(frame_df["knee_angle_asym_deg"].max())
    out["rule_max_hip_angle_asym_deg"] = float(frame_df["hip_angle_asym_deg"].max())
    out["rule_peak_valgus_score_global"] = float(frame_df["valgus_score_global"].max())
    out["rule_peak_hip_to_ankle_x_norm_femur"] = float(frame_df["hip_to_ankle_x_norm_femur"].max())
    out["rule_peak_knee_to_ankle_x_norm_tibia"] = float(frame_df["knee_to_ankle_x_norm_tibia"].max())


##GIVE CITATIONS
    out["possible_excessive_torso_lean"] = int(out["rule_peak_torso_lean_deg"] > 45)
    out["possible_limited_depth"] = int(out["rule_min_mean_knee_angle_deg"] > 100)
    out["possible_knee_angle_asymmetry"] = int(out["rule_max_knee_angle_asym_deg"] > 15)
    out["possible_valgus_pattern"] = int(out["rule_peak_valgus_score_global"] > 0.20)

    ###POSSIBLE WINK PROXY??

    return out


def compute_clip_level_features(points_seq):
    frame_rows = [extract_frame_features(p) for p in points_seq]
    frame_df = pd.DataFrame(frame_rows)

    clip_features = {"num_pose_frames": len(frame_df)}

    summary_cols = [
        "mean_knee_angle_deg",
        "knee_angle_asym_deg",
        "mean_hip_angle_deg",
        "hip_angle_asym_deg",
        "torso_lean_deg",
        "femur_tibia_ratio",
        "torso_femur_ratio",
        "hip_to_ankle_x_norm_femur",
        "knee_to_ankle_x_norm_tibia",
        "valgus_score_global",
        "valgus_asymmetry",
        "stance_width_norm_torso",
    ]

    for col in summary_cols:
        if col in frame_df.columns:
            clip_features.update(summarize_series(frame_df[col], col))

    dynamic_cols = [
        "mean_knee_angle_deg",
        "mean_hip_angle_deg",
        "torso_lean_deg",
        "hip_to_ankle_x_norm_femur",
        "knee_to_ankle_x_norm_tibia",
        "valgus_score_global",
    ]

    for col in dynamic_cols:
        if col in frame_df.columns:
            clip_features.update(compute_dynamic_summary(frame_df, col))

    if "mean_knee_angle_deg" in frame_df.columns and len(frame_df) > 0:
        bottom_idx = int(frame_df["mean_knee_angle_deg"].idxmin())
        bottom = frame_df.loc[bottom_idx]
        for col in [
            "torso_lean_deg",
            "hip_to_ankle_x_norm_femur",
            "knee_to_ankle_x_norm_tibia",
            "valgus_score_global",
            "femur_tibia_ratio",
            "torso_femur_ratio",
        ]:
            if col in bottom.index:
                clip_features[f"bottom_{col}"] = float(bottom[col])

    rule_measurements = compute_rule_based_measurements(frame_df)

    return clip_features, frame_df, rule_measurements

###potential final outputs?

PROBLEM: most common and pervasive issue may not be detected by MP joints -- anterior pelvic tilt -> wink

potential solution: find effects of that with other joints and with a combination of input conditions being met, label the issue as an output?



*   knee valgus (duple? R/L/both and severity score)
*   hips leading non-linearly (circular?)
*   



##building feature table from registry clips

In [ ]:


def build_feature_table_from_registry(only_usable=True, save=True):
    registry_df = load_master_index().copy()

    if only_usable and "usable" in registry_df.columns:
        registry_df = registry_df[registry_df["usable"].astype(str) != "0"]

    feature_rows = []
    rule_rows = []
    failed_rows = []

    for _, row in registry_df.iterrows():
        clip_id = row["clip_id"]
        vector_path = CLIPS_DIR / clip_id / "pose_vectors.csv"

        if not vector_path.exists():
            failed_rows.append({"clip_id": clip_id, "error": "missing pose_vectors.csv"})
            continue

        try:
            points_seq, raw_vec_df = load_vector_csv_as_points(vector_path)
            clip_features, frame_df, rules = compute_clip_level_features(points_seq)

            feature_rows.append({
                "clip_id": clip_id,
                "vector_csv": str(vector_path),
                **clip_features,
            })

            rule_rows.append({
                "clip_id": clip_id,
                **rules,
            })

        except Exception as e:
            failed_rows.append({"clip_id": clip_id, "error": str(e)})

    features_df = pd.DataFrame(feature_rows)
    rules_df = pd.DataFrame(rule_rows)
    failed_df = pd.DataFrame(failed_rows)

    if save:
        features_df.to_csv(ARTIFACT_DIR / "clip_features.csv", index=False)
        rules_df.to_csv(ARTIFACT_DIR / "rule_measurements.csv", index=False)
        failed_df.to_csv(ARTIFACT_DIR / "feature_build_failures.csv", index=False)

    print("Feature table:", features_df.shape)
    print("Rule table:", rules_df.shape)
    print("Failures:", failed_df.shape)

    display(features_df.head())
    display(rules_df.head())

    if len(failed_df):
        display(failed_df.head(20))

    return features_df, rules_df, failed_df


#run after pose_vectors.csv files exist:
features_df, rules_df, failed_df = build_feature_table_from_registry()

Feature table: (110, 141)
Rule table: (110, 12)
Failures: (0, 0)


,clip_id,vector_csv,num_pose_frames,mean_knee_angle_deg_mean,mean_knee_angle_deg_std,mean_knee_angle_deg_min,mean_knee_angle_deg_max,mean_knee_angle_deg_range,mean_knee_angle_deg_q25,mean_knee_angle_deg_median,...,valgus_score_global_velocity_mean_abs,valgus_score_global_velocity_max_abs,valgus_score_global_accel_mean_abs,valgus_score_global_accel_max_abs,bottom_torso_lean_deg,bottom_hip_to_ankle_x_norm_femur,bottom_knee_to_ankle_x_norm_tibia,bottom_valgus_score_global,bottom_femur_tibia_ratio,bottom_torso_femur_ratio
0,clip_0001,/content/drive/MyDrive/physio_project_updated2...,42,128.545532,41.899540,33.728577,177.943634,144.215057,97.068565,139.197845,...,4.073517,49.596939,7.586508,97.699921,123.607979,0.093828,0.979041,-0.072323,1.383307,1.663188
1,clip_0002,/content/drive/MyDrive/physio_project_updated2...,20,96.020615,30.835663,49.456524,147.428757,97.972229,66.089386,91.342346,...,2.623474,9.432848,4.568773,17.988583,147.629898,0.143831,0.785171,-0.690274,0.940774,1.353879
2,clip_0003,/content/drive/MyDrive/physio_project_updated2...,20,169.742950,8.214540,138.849152,175.752960,36.903809,170.910645,172.190140,...,0.353076,0.986717,0.593577,1.409022,128.794205,0.181610,0.424255,-0.244119,1.000846,1.429455
3,clip_0004,/content/drive/MyDrive/physio_project_updated2...,30,174.727051,9.439967,124.785309,179.961685,55.176376,174.793808,176.415482,...,0.070341,0.987157,0.089358,0.827559,104.325569,1.134056,0.909452,0.898962,0.926106,1.750755
4,clip_0005,/content/drive/MyDrive/physio_project_updated2...,25,124.196442,38.098526,58.993126,170.102371,111.109245,91.600311,134.908142,...,0.208656,0.472285,0.355249,0.898467,153.698517,0.547050,0.348531,-0.529820,0.796269,1.508428


,clip_id,rule_peak_torso_lean_deg,rule_min_mean_knee_angle_deg,rule_max_knee_angle_asym_deg,rule_max_hip_angle_asym_deg,rule_peak_valgus_score_global,rule_peak_hip_to_ankle_x_norm_femur,rule_peak_knee_to_ankle_x_norm_tibia,possible_excessive_torso_lean,possible_limited_depth,possible_knee_angle_asymmetry,possible_valgus_pattern
0,clip_0001,177.916611,33.728579,73.096886,46.340576,0.535381,1.743401,0.994840,1,0,1,1
1,clip_0002,179.871796,49.456524,71.598343,57.225616,0.582833,1.000841,0.789126,1,0,1,1
2,clip_0003,171.779678,138.849144,37.617172,22.008614,0.268426,0.695756,0.436576,1,1,1,1
3,clip_0004,179.965729,124.785309,24.635193,16.974838,0.898962,1.134056,0.909452,1,1,1,1
4,clip_0005,178.990295,58.993126,28.644989,21.892143,0.026677,0.734485,0.394636,1,0,1,0


In [ ]:
label_cols = [
    "clip_id",
    "good_form",
    "view_profile",
    "exercise_type",
    "load_type",
    "label_confidence",
    "view_confidence",
    "usable",
]

labels_df = load_master_index()[[c for c in label_cols if c in load_master_index().columns]].copy()

features_df = features_df.merge(
    labels_df,
    on="clip_id",
    how="left"
)

print(features_df.columns)
display(features_df.head())

Index(['clip_id', 'vector_csv', 'num_pose_frames', 'mean_knee_angle_deg_mean',
       'mean_knee_angle_deg_std', 'mean_knee_angle_deg_min',
       'mean_knee_angle_deg_max', 'mean_knee_angle_deg_range',
       'mean_knee_angle_deg_q25', 'mean_knee_angle_deg_median',
       ...
       'bottom_valgus_score_global', 'bottom_femur_tibia_ratio',
       'bottom_torso_femur_ratio', 'good_form', 'view_profile',
       'exercise_type', 'load_type', 'label_confidence', 'view_confidence',
       'usable'],
      dtype='object', length=148)


,clip_id,vector_csv,num_pose_frames,mean_knee_angle_deg_mean,mean_knee_angle_deg_std,mean_knee_angle_deg_min,mean_knee_angle_deg_max,mean_knee_angle_deg_range,mean_knee_angle_deg_q25,mean_knee_angle_deg_median,...,bottom_valgus_score_global,bottom_femur_tibia_ratio,bottom_torso_femur_ratio,good_form,view_profile,exercise_type,load_type,label_confidence,view_confidence,usable
0,clip_0001,/content/drive/MyDrive/physio_project_updated2...,42,128.545532,41.899540,33.728577,177.943634,144.215057,97.068565,139.197845,...,-0.072323,1.383307,1.663188,0,left_side,back_squat,barbell,high,medium,1
1,clip_0002,/content/drive/MyDrive/physio_project_updated2...,20,96.020615,30.835663,49.456524,147.428757,97.972229,66.089386,91.342346,...,-0.690274,0.940774,1.353879,1,left_side,back_squat,barbell,high,high,1
2,clip_0003,/content/drive/MyDrive/physio_project_updated2...,20,169.742950,8.214540,138.849152,175.752960,36.903809,170.910645,172.190140,...,-0.244119,1.000846,1.429455,1,front_left,back_squat,barbell,medium,medium,1
3,clip_0004,/content/drive/MyDrive/physio_project_updated2...,30,174.727051,9.439967,124.785309,179.961685,55.176376,174.793808,176.415482,...,0.898962,0.926106,1.750755,0,left_side,back_squat,barbell,medium,medium,1
4,clip_0005,/content/drive/MyDrive/physio_project_updated2...,25,124.196442,38.098526,58.993126,170.102371,111.109245,91.600311,134.908142,...,-0.529820,0.796269,1.508428,1,front_left,back_squat,barbell,medium,high,1


In [ ]:
#train/test split by clip
#input: features_df from build_feature_table_from_registry()
#ouput: X_train, X_test, y_train, y_test, trainable_df, feature_cols

if "features_df" not in globals():
    raise NameError("features_df not found. Run the 'building feature table from registry clips' section first.")

required_cols = ["clip_id", "good_form"]
missing_cols = [c for c in required_cols if c not in features_df.columns]
if missing_cols:
    raise ValueError(f"features_df is missing required columns: {missing_cols}")

trainable_df = features_df.dropna(subset=["good_form"]).copy()
trainable_df["good_form"] = trainable_df["good_form"].astype(int)

#NOT MODEL INPUTS!!!! just for identity/explnation
metadata_cols = [
    c for c in [
        "clip_id",
        "source_csv",
        "good_form",
        "view_profile",
        "exercise_type",
        "load_type",
        "label_confidence",
        "view_confidence",
        "usable",
    ]
    if c in trainable_df.columns
]

explanation_cols = [
    c for c in trainable_df.columns
    if c.startswith("rule_") or c.startswith("possible_")
]

#using only numeric engineered columns for model inputs
excluded_cols = set(metadata_cols + explanation_cols)

feature_cols = [
    c for c in trainable_df.columns
    if c not in excluded_cols
    and pd.api.types.is_numeric_dtype(trainable_df[c])
]

if len(feature_cols) == 0:
    raise ValueError("No numeric feature columns found for modeling.")

X = trainable_df[feature_cols].copy()
y = trainable_df["good_form"].copy()

# stratifying only if both classes have enough examples
stratify_y = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=stratify_y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("\nTrain label counts:")
print(y_train.value_counts())
print("\nTest label counts:")
print(y_test.value_counts())

print(f"\nUsing {len(feature_cols)} numeric engineered feature columns.")
print("First 10 feature columns:")
print(feature_cols[:10])

Train: (82, 139)
Test: (28, 139)

Train label counts:
good_form
0    42
1    40
Name: count, dtype: int64

Test label counts:
good_form
0    15
1    13
Name: count, dtype: int64

Using 139 numeric engineered feature columns.
First 10 feature columns:
['num_pose_frames', 'mean_knee_angle_deg_mean', 'mean_knee_angle_deg_std', 'mean_knee_angle_deg_min', 'mean_knee_angle_deg_max', 'mean_knee_angle_deg_range', 'mean_knee_angle_deg_q25', 'mean_knee_angle_deg_median', 'mean_knee_angle_deg_q75', 'mean_knee_angle_deg_slope']


In [ ]:
#auditing possible string/object columns before modeling

if "features_df" not in globals():
    raise NameError("features_df not found. Run build_feature_table_from_registry() first.")

print("features_df shape:", features_df.shape)

object_cols = features_df.select_dtypes(include=["object", "string"]).columns.tolist()

print("\nString/object columns in features_df:")
print(object_cols)

print("\nSample values from string/object columns:")
for col in object_cols:
    print("\n" + "=" * 60)
    print(col)
    print(features_df[col].dropna().head(10).tolist())

features_df shape: (110, 148)

String/object columns in features_df:
['clip_id', 'vector_csv', 'view_profile', 'exercise_type', 'load_type', 'label_confidence', 'view_confidence']

Sample values from string/object columns:

clip_id
['clip_0001', 'clip_0002', 'clip_0003', 'clip_0004', 'clip_0005', 'clip_0006', 'clip_0007', 'clip_0008', 'clip_0009', 'clip_0010']

vector_csv
['/content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0001/pose_vectors.csv', '/content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0002/pose_vectors.csv', '/content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0003/pose_vectors.csv', '/content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0004/pose_vectors.csv', '/content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0005/pose_vectors.csv', '/content/drive/MyDrive/physio_project_updated2026_05_18/dataset/clips/clip_0006/pose_vectors.csv', '/content/drive/MyDrive/

In [ ]:
#checking time-related columns

time_like_cols = [
    c for c in features_df.columns
    if "time" in c.lower() or c in ["start_sec", "end_sec"]
]

print("Time-like columns:")
print(time_like_cols)

for col in time_like_cols:
    print("\n" + "=" * 60)
    print(col)
    print("dtype:", features_df[col].dtype)
    print(features_df[col].dropna().head(10).tolist())

Time-like columns:
[]


In [ ]:
#simulating current feature selection and catching non-numeric leaks

non_feature_cols = {
    "clip_id",
    "good_form",
    "source_csv",
    "vector_csv",
    "pose_csv",
    "url",
    "video_url",
    "start_time",
    "end_time",
    "view_profile",
    "exercise_type",
    "load_type",
    "label_confidence",
    "view_confidence",
    "usable",
}

non_feature_cols.update([
    c for c in features_df.columns
    if c.startswith("rule_") or c.startswith("possible_")
])

candidate_cols = [
    c for c in features_df.columns
    if c not in non_feature_cols
]

non_numeric_candidates = [
    c for c in candidate_cols
    if not pd.api.types.is_numeric_dtype(features_df[c])
]

print("Candidate model columns:", len(candidate_cols))
print("Non-numeric candidate columns that would break modeling:")
print(non_numeric_candidates)

Candidate model columns: 139
Non-numeric candidate columns that would break modeling:
[]


In [ ]:
#preparing supervised ML table from engineered features
#input: features_df from build_feature_table_from_registry()
#output: X_train, X_test, y_train, y_test, trainable_df, feature_cols

if "features_df" not in globals():
    raise NameError("features_df not found. Run build_feature_table_from_registry() first.")

if "good_form" not in features_df.columns:
    raise ValueError("features_df is missing good_form. Merge labels from master_index.csv first.")

trainable_df = features_df.dropna(subset=["good_form"]).copy()
trainable_df["good_form"] = trainable_df["good_form"].astype(int)

#keeping ID/metadata for inspection, DO NOT FEED TO MODEL !!!!!
metadata_cols = [
    c for c in trainable_df.columns
    if (
        c == "clip_id"
        or c == "good_form"
        or trainable_df[c].dtype == "object"
        or str(trainable_df[c].dtype).startswith("string")
    )
]

rule_cols = [
    c for c in trainable_df.columns
    if c.startswith("rule_") or c.startswith("possible_")
]

excluded_cols = set(metadata_cols + rule_cols)

#ONLY NUMERIC FEATURES ENTER X
feature_cols = [
    c for c in trainable_df.columns
    if c not in excluded_cols
    and pd.api.types.is_numeric_dtype(trainable_df[c])
]

if len(feature_cols) == 0:
    raise ValueError("No numeric engineered feature columns found.")

X = trainable_df[feature_cols].copy()
y = trainable_df["good_form"].copy()

#sanity checks
non_numeric_in_X = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_in_X:
    raise TypeError(f"Non-numeric columns leaked into X: {non_numeric_in_X}")

print("Total rows available:", len(trainable_df))
print("Model feature count:", len(feature_cols))
print("Metadata/string columns excluded:", metadata_cols)
print("Rule/explanation columns excluded:", rule_cols)

print("\nLabel counts:")
print(y.value_counts())

stratify_y = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=stratify_y
)

print("\nTrain:", X_train.shape)
print("Test:", X_test.shape)
print("\nFirst 15 model features:")
print(feature_cols[:15])

Total rows available: 110
Model feature count: 140
Metadata/string columns excluded: ['clip_id', 'vector_csv', 'good_form', 'view_profile', 'exercise_type', 'load_type', 'label_confidence', 'view_confidence']
Rule/explanation columns excluded: []

Label counts:
good_form
0    57
1    53
Name: count, dtype: int64

Train: (82, 140)
Test: (28, 140)

First 15 model features:
['num_pose_frames', 'mean_knee_angle_deg_mean', 'mean_knee_angle_deg_std', 'mean_knee_angle_deg_min', 'mean_knee_angle_deg_max', 'mean_knee_angle_deg_range', 'mean_knee_angle_deg_q25', 'mean_knee_angle_deg_median', 'mean_knee_angle_deg_q75', 'mean_knee_angle_deg_slope', 'knee_angle_asym_deg_mean', 'knee_angle_asym_deg_std', 'knee_angle_asym_deg_min', 'knee_angle_asym_deg_max', 'knee_angle_asym_deg_range']


##training baseline CNN models

In [ ]:
def prepare_model_dataset(features_df, rules_df=None, registry_df=None):
    if registry_df is None:
        registry_df = load_master_index()

    df = features_df.copy()

    #avoiding duplicate label columns from previous messy merges
    duplicate_label_cols = [
        c for c in df.columns
        if c in ["good_form", "view_profile", "exercise_type", "load_type",
                 "label_confidence", "view_confidence", "usable"]
    ]
    df = df.drop(columns=duplicate_label_cols, errors="ignore")

    label_cols = [
        "clip_id",
        "good_form",
        "view_profile",
        "exercise_type",
        "load_type",
        "label_confidence",
        "view_confidence",
        "usable",
    ]
    label_cols = [c for c in label_cols if c in registry_df.columns]

    df = df.merge(
        registry_df[label_cols],
        on="clip_id",
        how="left"
    )

    if rules_df is not None:
        rule_cols_to_drop = [
            c for c in df.columns
            if c.startswith("rule_") or c.startswith("possible_")
        ]
        df = df.drop(columns=rule_cols_to_drop, errors="ignore")
        df = df.merge(rules_df, on="clip_id", how="left")

    if "good_form" not in df.columns:
        raise ValueError("Could not attach good_form labels.")

    df = df.dropna(subset=["good_form"]).copy()
    df["good_form"] = df["good_form"].astype(int)

    rule_cols = [
        c for c in df.columns
        if c.startswith("rule_") or c.startswith("possible_")
    ]

    metadata_cols = [
        c for c in df.columns
        if (
            c == "clip_id"
            or c == "good_form"
            or df[c].dtype == "object"
            or str(df[c].dtype).startswith("string")
        )
    ]

    excluded_cols = set(metadata_cols + rule_cols)

    feature_cols = [
        c for c in df.columns
        if c not in excluded_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    X = df[feature_cols].copy()
    y = df["good_form"].copy()

    return df, X, y, feature_cols, rule_cols

In [ ]:
def score_movement_contributors(row):
    contributors = []

    def add(name, score, reason):
        score = float(np.clip(score, 0, 1))
        if score >= 0.67:
            level = "elevated"
        elif score >= 0.34:
            level = "moderate"
        else:
            level = "low"

        contributors.append({
            "issue": name,
            "score": score,
            "level": level,
            "reason": reason,
        })

    #knee valgus/collapse
    valgus = row.get("rule_peak_valgus_score_global", np.nan)
    if not pd.isna(valgus):
        add(
            "knee_valgus_pattern",
            valgus / 0.35,
            f"Peak valgus score was {valgus:.3f}."
        )

    #limited depth
    min_knee = row.get("rule_min_mean_knee_angle_deg", np.nan)
    if not pd.isna(min_knee):
        # Higher minimum knee angle means less depth
        add(
            "limited_depth",
            (min_knee - 70) / 50,
            f"Minimum mean knee angle was {min_knee:.1f}°."
        )

    #torso lean
    torso = row.get("rule_peak_torso_lean_deg", np.nan)
    if not pd.isna(torso):
        add(
            "excessive_torso_lean",
            (torso - 25) / 35,
            f"Peak torso lean was {torso:.1f}°."
        )

    #asymmetry
    knee_asym = row.get("rule_max_knee_angle_asym_deg", np.nan)
    hip_asym = row.get("rule_max_hip_angle_asym_deg", np.nan)

    asym_values = [v for v in [knee_asym, hip_asym] if not pd.isna(v)]
    if asym_values:
        asym = max(asym_values)
        add(
            "left_right_asymmetry",
            asym / 30,
            f"Maximum knee/hip asymmetry was {asym:.1f}°."
        )

    contributors = sorted(contributors, key=lambda d: d["score"], reverse=True)
    return contributors

In [ ]:
def train_engineered_feature_models(features_df, rules_df=None, registry_df=None, test_size=0.25, random_state=42):
    dataset_df, X, y, feature_cols, rule_cols = prepare_model_dataset(
        features_df=features_df,
        rules_df=rules_df,
        registry_df=registry_df
    )

    if y.nunique() < 2:
        raise ValueError("Need both good_form classes before training.")

    stratify_y = y if y.value_counts().min() >= 2 else None

    X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
        X,
        y,
        dataset_df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_y
    )

    models = {
        "logreg": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state)),
        ]),
        "random_forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=random_state)),
        ]),
        "gradient_boosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GradientBoostingClassifier(random_state=random_state)),
        ]),
    }

    results = []
    fitted_models = {}

    for name, model in models.items():
        fitted = clone(model)
        fitted.fit(X_train, y_train)

        y_pred = fitted.predict(X_test)

        results.append({
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
        })

        fitted_models[name] = fitted

    results_df = pd.DataFrame(results).sort_values(
        ["balanced_accuracy", "f1", "accuracy"],
        ascending=False
    ).reset_index(drop=True)

    best_model_name = results_df.iloc[0]["model"]
    best_model = fitted_models[best_model_name]

    y_pred_best = best_model.predict(X_test)

    test_output_df = dataset_df.loc[test_idx].copy()
    test_output_df["pred_good_form"] = y_pred_best

    if hasattr(best_model, "predict_proba"):
        test_output_df["good_form_probability"] = best_model.predict_proba(X_test)[:, 1]
        test_output_df["needs_attention_probability"] = 1 - test_output_df["good_form_probability"]

    test_output_df["movement_contributors"] = test_output_df.apply(
        lambda row: score_movement_contributors(row),
        axis=1
    )

    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)
    display(results_df)

    print("\nBest model:", best_model_name)
    print(classification_report(y_test, y_pred_best, digits=3, zero_division=0))
    print(confusion_matrix(y_test, y_pred_best))

    return {
        "dataset_df": dataset_df,
        "results_df": results_df,
        "fitted_models": fitted_models,
        "best_model_name": best_model_name,
        "best_model": best_model,
        "feature_cols": feature_cols,
        "rule_cols": rule_cols,
        "test_output_df": test_output_df,
        "X_test": X_test,
        "y_test": y_test,
    }

In [ ]:
features_df, rules_df, failed_df = build_feature_table_from_registry()

train_outputs = train_engineered_feature_models(features_df, rules_df)

display(train_outputs["test_output_df"][[
    "clip_id",
    "good_form",
    "pred_good_form",
    "good_form_probability",
    "needs_attention_probability",
    "movement_contributors"
]].head())

Feature table: (110, 141)
Rule table: (110, 12)
Failures: (0, 0)


,clip_id,vector_csv,num_pose_frames,mean_knee_angle_deg_mean,mean_knee_angle_deg_std,mean_knee_angle_deg_min,mean_knee_angle_deg_max,mean_knee_angle_deg_range,mean_knee_angle_deg_q25,mean_knee_angle_deg_median,...,valgus_score_global_velocity_mean_abs,valgus_score_global_velocity_max_abs,valgus_score_global_accel_mean_abs,valgus_score_global_accel_max_abs,bottom_torso_lean_deg,bottom_hip_to_ankle_x_norm_femur,bottom_knee_to_ankle_x_norm_tibia,bottom_valgus_score_global,bottom_femur_tibia_ratio,bottom_torso_femur_ratio
0,clip_0001,/content/drive/MyDrive/physio_project_updated2...,42,128.545532,41.899540,33.728577,177.943634,144.215057,97.068565,139.197845,...,4.073517,49.596939,7.586508,97.699921,123.607979,0.093828,0.979041,-0.072323,1.383307,1.663188
1,clip_0002,/content/drive/MyDrive/physio_project_updated2...,20,96.020615,30.835663,49.456524,147.428757,97.972229,66.089386,91.342346,...,2.623474,9.432848,4.568773,17.988583,147.629898,0.143831,0.785171,-0.690274,0.940774,1.353879
2,clip_0003,/content/drive/MyDrive/physio_project_updated2...,20,169.742950,8.214540,138.849152,175.752960,36.903809,170.910645,172.190140,...,0.353076,0.986717,0.593577,1.409022,128.794205,0.181610,0.424255,-0.244119,1.000846,1.429455
3,clip_0004,/content/drive/MyDrive/physio_project_updated2...,30,174.727051,9.439967,124.785309,179.961685,55.176376,174.793808,176.415482,...,0.070341,0.987157,0.089358,0.827559,104.325569,1.134056,0.909452,0.898962,0.926106,1.750755
4,clip_0005,/content/drive/MyDrive/physio_project_updated2...,25,124.196442,38.098526,58.993126,170.102371,111.109245,91.600311,134.908142,...,0.208656,0.472285,0.355249,0.898467,153.698517,0.547050,0.348531,-0.529820,0.796269,1.508428


,clip_id,rule_peak_torso_lean_deg,rule_min_mean_knee_angle_deg,rule_max_knee_angle_asym_deg,rule_max_hip_angle_asym_deg,rule_peak_valgus_score_global,rule_peak_hip_to_ankle_x_norm_femur,rule_peak_knee_to_ankle_x_norm_tibia,possible_excessive_torso_lean,possible_limited_depth,possible_knee_angle_asymmetry,possible_valgus_pattern
0,clip_0001,177.916611,33.728579,73.096886,46.340576,0.535381,1.743401,0.994840,1,0,1,1
1,clip_0002,179.871796,49.456524,71.598343,57.225616,0.582833,1.000841,0.789126,1,0,1,1
2,clip_0003,171.779678,138.849144,37.617172,22.008614,0.268426,0.695756,0.436576,1,1,1,1
3,clip_0004,179.965729,124.785309,24.635193,16.974838,0.898962,1.134056,0.909452,1,1,1,1
4,clip_0005,178.990295,58.993126,28.644989,21.892143,0.026677,0.734485,0.394636,1,0,1,0


Train shape: (82, 140)
Test shape: (28, 140)


,model,accuracy,balanced_accuracy,f1,precision,recall
0,random_forest,0.642857,0.641026,0.615385,0.615385,0.615385
1,logreg,0.571429,0.574359,0.571429,0.533333,0.615385
2,gradient_boosting,0.535714,0.525641,0.434783,0.500000,0.384615



Best model: random_forest
              precision    recall  f1-score   support

           0      0.667     0.667     0.667        15
           1      0.615     0.615     0.615        13

    accuracy                          0.643        28
   macro avg      0.641     0.641     0.641        28
weighted avg      0.643     0.643     0.643        28

[[10  5]
 [ 5  8]]


,clip_id,good_form,pred_good_form,good_form_probability,needs_attention_probability,movement_contributors
44,clip_0045,0,0,0.343333,0.656667,"[{'issue': 'knee_valgus_pattern', 'score': 1.0..."
108,clip_0109,0,1,0.620000,0.380000,"[{'issue': 'excessive_torso_lean', 'score': 1...."
102,clip_0103,0,1,0.513333,0.486667,"[{'issue': 'knee_valgus_pattern', 'score': 1.0..."
96,clip_0097,0,0,0.460000,0.540000,"[{'issue': 'knee_valgus_pattern', 'score': 1.0..."
88,clip_0089,0,0,0.473333,0.526667,"[{'issue': 'knee_valgus_pattern', 'score': 1.0..."


##saving artifacts

In [ ]:


def save_mvp_artifacts(train_outputs, features_df, rules_df):
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    best_model_name = train_outputs["best_model_name"]
    best_model = train_outputs["best_model"]

    model_path = MODEL_DIR / f"{best_model_name}_best_model.joblib"
    joblib.dump(best_model, model_path)

    schema_path = MODEL_DIR / "feature_schema.json"
    with open(schema_path, "w") as f:
        json.dump(train_outputs["feature_schema"], f, indent=2)

    features_df.to_csv(ARTIFACT_DIR / "clip_features.csv", index=False)
    rules_df.to_csv(ARTIFACT_DIR / "rule_measurements.csv", index=False)
    train_outputs["dataset_df"].to_csv(ARTIFACT_DIR / "dataset_with_labels_and_rules.csv", index=False)
    train_outputs["results_df"].to_csv(ARTIFACT_DIR / "model_results.csv", index=False)
    train_outputs["test_predictions_df"].to_csv(ARTIFACT_DIR / "test_predictions.csv", index=False)

    print("Saved model:", model_path)
    print("Saved schema:", schema_path)
    print("Saved feature artifacts to:", ARTIFACT_DIR)


#EX:
#save_mvp_artifacts(train_outputs, features_df, rules_df)

##save reusable processed dataset zip


In [ ]:


def zip_clip_folders(zip_name="clips_with_vectors.zip"):
    zip_path = VECTORS_ZIP_DIR / zip_name

    if zip_path.exists():
        zip_path.unlink()

    shutil.make_archive(
        base_name=str(zip_path).replace(".zip", ""),
        format="zip",
        root_dir=str(CLIPS_DIR),
    )

    print("Saved zip:", zip_path)
    return zip_path


#zip_clip_folders()